# A3 — Speech model comparison for Basira (Listen stage)

**Purpose:** choose the acoustic model for per-phoneme scoring (issue #3). Compares two Apache-2.0 models on the
contrasts Basira depends on: Kuwaiti dialect variants (ق→[g], ج→[j]) and Arabic error categories (dot and emphatic
confusions).

| Model | Output | Licence |
| --- | --- | --- |
| `facebook/wav2vec2-xlsr-53-espeak-cv-ft` | phonemes (espeak IPA) | Apache-2.0 |
| `jonatasgrosman/wav2vec2-large-xlsr-53-arabic` | Arabic letters | Apache-2.0 |

**Everything here is SYNTHETIC.** Test audio is adult neural TTS (Microsoft Edge voices via the `edge-tts` client) —
used only for this internal comparison, never shipped. It is not child speech; results show whether a model *can*
represent a contrast, not how it performs on children.

**How to run:** Runtime → Change runtime type → T4 GPU (optional, CPU also works) → Runtime → Run all.
Then copy the block printed by the last cell and paste it back to Claude.

In [ ]:
%%capture
!apt-get -qq install -y espeak-ng > /dev/null
!pip -q install -U edge-tts gTTS phonemizer==3.3.0
# transformers, torch, librosa, soundfile: use the versions preinstalled in Colab

In [ ]:
import asyncio, json, os, unicodedata, numpy as np, torch, librosa, edge_tts
from transformers import AutoProcessor, AutoModelForCTC
from phonemizer.backend import EspeakBackend
from phonemizer.separator import Separator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
os.makedirs("audio", exist_ok=True)
HARAKAT = "".join(chr(c) for c in range(0x064B, 0x0653)) + "ٰ"
strip = lambda s: "".join(ch for ch in s if ch not in HARAKAT)

## Test cases

Each case synthesizes `spoken` with a TTS voice and scores it against the **expected** (correct MSA) word.
`hypothesis` is what a good model should show at the target position.

Dialect audio trick: Egyptian voices pronounce ج as [g], so `جَلَم` in an Egyptian voice approximates Kuwaiti
`قَلَم` said as [galam].

In [ ]:
CASES = [
  # id, expected (MSA, diacritized), spoken text, voice, target letter in expected, ipa expected, ipa if child did as hypothesised, hypothesis
  dict(id="qaaf_msa",      expected="قَلَم",  spoken="قَلَم",  voice="ar-SA-HamedNeural",  letter="ق", exp_ph="q",  alt_ph="ɡ",  hyp="correct: q"),
  dict(id="qaaf_as_g",     expected="قَلَم",  spoken="جَلَم",  voice="ar-EG-ShakirNeural", letter="ق", exp_ph="q",  alt_ph="ɡ",  hyp="dialect: model hears ɡ, not q"),
  dict(id="thaa_msa",      expected="ثَوْب",  spoken="ثَوْب",  voice="ar-SA-HamedNeural",  letter="ث", exp_ph="θ",  alt_ph="t",  hyp="correct: θ"),
  dict(id="thaa_as_taa",   expected="ثَوْب",  spoken="تَوْب",  voice="ar-SA-HamedNeural",  letter="ث", exp_ph="θ",  alt_ph="t",  hyp="dot confusion: hears t"),
  dict(id="dhaal_as_daal", expected="ذَهَب",  spoken="دَهَب",  voice="ar-SA-HamedNeural",  letter="ذ", exp_ph="ð",  alt_ph="d",  hyp="dot confusion: hears d"),
  dict(id="khaa_as_hhaa",  expected="خَرُوف", spoken="حَرُوف", voice="ar-SA-HamedNeural",  letter="خ", exp_ph="x",  alt_ph="ħ",  hyp="dot confusion: hears ħ"),
  dict(id="saad_msa",      expected="صَبَاح", spoken="صَبَاح", voice="ar-SA-HamedNeural",  letter="ص", exp_ph="s^", alt_ph="s",  hyp="correct: emphatic s"),
  dict(id="saad_as_siin",  expected="صَبَاح", spoken="سَبَاح", voice="ar-SA-HamedNeural",  letter="ص", exp_ph="s^", alt_ph="s",  hyp="emphatic confusion: hears plain s"),
  dict(id="jiim_as_y",     expected="دَجَاج", spoken="دَيَاي", voice="ar-KW-FahedNeural",  letter="ج", exp_ph="dʒ", alt_ph="j",  hyp="dialect: model hears j, not dʒ"),
  dict(id="kw_voice_qalam",expected="قَلَم",  spoken="قَلَم",  voice="ar-KW-FahedNeural",  letter="ق", exp_ph="q",  alt_ph="ɡ",  hyp="exploratory: what a Kuwaiti TTS voice does"),
]

async def synth(case):
    # Microsoft neural voices via edge-tts (needed for the dialect cases); Google TTS as a fallback.
    path = f"audio/{case['id']}.mp3"
    case["tts"] = "edge-tts:" + case["voice"]
    if os.path.exists(path):
        return path
    for attempt in range(3):
        try:
            await edge_tts.Communicate(case["spoken"], case["voice"]).save(path)
            return path
        except Exception as e:
            err = e
            await asyncio.sleep(2)
    from gtts import gTTS
    gTTS(case["spoken"], lang="ar").save(path)
    case["tts"] = f"gTTS-ar FALLBACK (edge-tts failed: {type(err).__name__})"
    return path

for c in CASES:
    c["path"] = await synth(c)
    print(c["id"].ljust(16), c["tts"])
print("synthesized", len(CASES), "clips")

## Models and CTC forced alignment

In [ ]:
def load(name):
    proc = AutoProcessor.from_pretrained(name)
    model = AutoModelForCTC.from_pretrained(name).to(DEVICE).eval()
    vocab = proc.tokenizer.get_vocab()
    return proc, model, vocab

def log_posteriors(proc, model, path):
    wav, _ = librosa.load(path, sr=16000)
    inp = proc(wav, sampling_rate=16000, return_tensors="pt").input_values.to(DEVICE)
    with torch.no_grad():
        logits = model(inp).logits[0]
    return torch.log_softmax(logits, -1).cpu().numpy()      # (T, V)

def ctc_align(lp, targets, blank):
    # Viterbi CTC forced alignment. Returns list of (start, end) frame spans per target token.
    T, L = lp.shape[0], len(targets)
    ext = [blank]
    for t in targets: ext += [t, blank]
    S = len(ext); NEG = -1e9
    dp = np.full((T, S), NEG); bp = np.zeros((T, S), dtype=int)
    dp[0, 0] = lp[0, ext[0]]
    if S > 1: dp[0, 1] = lp[0, ext[1]]
    for t in range(1, T):
        for s in range(S):
            cands = [(dp[t-1, s], s)]
            if s >= 1: cands.append((dp[t-1, s-1], s-1))
            if s >= 2 and ext[s] != blank and ext[s] != ext[s-2]: cands.append((dp[t-1, s-2], s-2))
            best, arg = max(cands)
            dp[t, s] = best + lp[t, ext[s]]; bp[t, s] = arg
    s = S-1 if S < 2 or dp[T-1, S-1] >= dp[T-1, S-2] else S-2
    path = []
    for t in range(T-1, -1, -1):
        path.append(s); s = bp[t, s]
    path.reverse()
    spans = [[None, None] for _ in range(L)]
    for t, s in enumerate(path):
        if s % 2 == 1:
            i = s // 2
            if spans[i][0] is None: spans[i][0] = t
            spans[i][1] = t + 1
    return spans

def segment_scores(lp, span, ids):
    a, b = span
    return {tok: float(lp[a:b, i].mean()) for tok, i in ids.items()}

def greedy(lp, inv, blank):
    best = lp.argmax(-1); out = []; prev = None
    for k in best:
        if k != prev and k != blank: out.append(inv[k])
        prev = k
    return out

In [ ]:
# ---- phoneme model ----
PH = "facebook/wav2vec2-xlsr-53-espeak-cv-ft"
ph_proc, ph_model, ph_vocab = load(PH)
ph_inv = {v: k for k, v in ph_vocab.items()}
ph_blank = ph_proc.tokenizer.pad_token_id
espeak = EspeakBackend("ar", preserve_punctuation=False, with_stress=False)
SEP = Separator(phone=" ", word=" | ", syllable="")

def to_ph_tokens(text):
    raw = espeak.phonemize([text], separator=SEP, strip=True)[0]
    toks, unknown = [], []
    for p in raw.replace("|", " ").split():
        if p in ph_vocab: toks.append(p); continue
        # greedy longest-match split for multi-char espeak symbols
        i = 0
        while i < len(p):
            for j in range(len(p), i, -1):
                if p[i:j] in ph_vocab: toks.append(p[i:j]); i = j; break
            else:
                unknown.append(p[i]); i += 1
    return raw, toks, unknown

# ---- letter model ----
GR = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"
gr_proc, gr_model, gr_vocab = load(GR)
gr_inv = {v: k for k, v in gr_vocab.items()}
gr_blank = gr_proc.tokenizer.pad_token_id
print("loaded both models")

In [ ]:
results = []
for c in CASES:
    r = dict(id=c["id"], hyp=c["hyp"], expected=c["expected"], spoken=c["spoken"], voice=c["voice"], tts=c["tts"])

    # phoneme model: align expected espeak phones; compare expected vs alternative phone at the target
    raw, toks, unk = to_ph_tokens(c["expected"])
    lp = log_posteriors(ph_proc, ph_model, c["path"])
    r["ph_expected_raw"] = raw; r["ph_unknown"] = unk
    r["ph_heard_free"] = " ".join(greedy(lp, ph_inv, ph_blank))
    if c["exp_ph"] in toks and c["alt_ph"] in ph_vocab:
        spans = ctc_align(lp, [ph_vocab[t] for t in toks], ph_blank)
        k = toks.index(c["exp_ph"])
        sc = segment_scores(lp, spans[k], {c["exp_ph"]: ph_vocab[c["exp_ph"]], c["alt_ph"]: ph_vocab[c["alt_ph"]]})
        r["ph_target_logp"] = round(sc[c["exp_ph"]], 2); r["ph_alt_logp"] = round(sc[c["alt_ph"]], 2)
        r["ph_prefers"] = c["exp_ph"] if sc[c["exp_ph"]] >= sc[c["alt_ph"]] else c["alt_ph"]
    else:
        r["ph_prefers"] = f"n/a (expected phones {toks}; target {c['exp_ph']!r} or alt {c['alt_ph']!r} missing)"

    # letter model: align expected letters; report top letters at the target position
    letters = [ch for ch in strip(c["expected"]) if ch in gr_vocab]
    lp2 = log_posteriors(gr_proc, gr_model, c["path"])
    spans2 = ctc_align(lp2, [gr_vocab[ch] for ch in letters], gr_blank)
    k2 = letters.index(c["letter"])
    a, b = spans2[k2]
    mean = lp2[a:b].mean(0)
    top = [gr_inv[i] for i in np.argsort(-mean)[:3] if i != gr_blank]
    r["gr_heard_free"] = "".join(greedy(lp2, gr_inv, gr_blank))
    r["gr_top_at_target"] = top
    results.append(r)
print("scored", len(results), "cases")

In [ ]:
# ===== COPY EVERYTHING BELOW THIS LINE AND PASTE IT TO CLAUDE =====
print("A3-MODEL-COMPARISON v2 (SYNTHETIC TTS AUDIO)")
import transformers; print("python", __import__("sys").version.split()[0], "| transformers", transformers.__version__, "| torch", torch.__version__)
print("device", DEVICE)
for r in results:
    print("---", r["id"], "|", r["hyp"])
    print("  expected", r["expected"], "spoken", r["spoken"], "| tts:", r["tts"])
    print("  PH espeak:", r["ph_expected_raw"], "| unknown:", r["ph_unknown"])
    print("  PH heard :", r["ph_heard_free"])
    print("  PH target vs alt logp:", r.get("ph_target_logp"), r.get("ph_alt_logp"), "-> prefers", r["ph_prefers"])
    print("  GR heard :", r["gr_heard_free"], "| top letters at target:", r["gr_top_at_target"])
json.dump(results, open("a3_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)